In [ ]:
!pip install kagglehub -q

import kagglehub
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

In [ ]:
print("Đang tải dataset...")
path = kagglehub.dataset_download("nguyentrongdai/vietnamese-currency")
print("Đã tải xong! Đường dẫn thư mục:", path)

Đang tải dataset...
Using Colab cache for faster access to the 'vietnamese-currency' dataset.
Đã tải xong! Đường dẫn thư mục: /kaggle/input/vietnamese-currency


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_PATH = "/kaggle/input/vietnamese-currency/dataset"


train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)


train_generator = train_datagen.flow_from_directory(
    DATA_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    DATA_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 2172 images belonging to 12 classes.
Found 540 images belonging to 12 classes.


In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(12, activation='softmax')
])

In [ ]:

labels = train_generator.class_indices
print(labels)

{'000000': 0, '000200': 1, '000500': 2, '001000': 3, '002000': 4, '005000': 5, '010000': 6, '020000': 7, '050000': 8, '100000': 9, '200000': 10, '500000': 11}


In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
print("Bắt đầu huấn luyện mô hình...")
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=15
)

Bắt đầu huấn luyện mô hình...
Epoch 1/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 168s 2s/step - accuracy: 0.3352 - loss: 2.0169 - val_accuracy: 0.4241 - val_loss: 1.6894
Epoch 2/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 168s 2s/step - accuracy: 0.5483 - loss: 1.3499 - val_accuracy: 0.5407 - val_loss: 1.3760
Epoch 3/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 160s 2s/step - accuracy: 0.6648 - loss: 1.0422 - val_accuracy: 0.5537 - val_loss: 1.2974
Epoch 4/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 154s 2s/step - accuracy: 0.6952 - loss: 0.9296 - val_accuracy: 0.6000 - val_loss: 1.2002
Epoch 5/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 153s 2s/step - accuracy: 0.7445 - loss: 0.7794 - val_accuracy: 0.6333 - val_loss: 1.1040
Epoch 6/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 145s 2s/step - accuracy: 0.7739 - loss: 0.7059 - val_accuracy: 0.6519 - val_loss: 1.0517
Epoch 7/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 160s 2s/step - accuracy: 0.7753 - loss: 0.6649 - val_accuracy: 0.6444 - val_loss: 0.9955
Epoch 8/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 158s 2s/step - accuracy: 0.7942 - loss: 0.6

In [ ]:
model.save('vietnamese_money_v1.h5')